# DR retraining - COMPREHENSIVE (APTOS 2019): full arm matrix + reproduction attempts + reviewer analyses
Runtime > Change runtime type > **GPU (A100)**. Every experiment is resumable (finished runs are skipped) and mirrored to Drive.

| Exp | What | Why |
|---|---|---|
| E0_main | split 2929/366/367, 12 arms x 3 seeds (+analysis) | gate vs equal, T1/T2 roles, augmentation on the FINAL model, latency, calibration, paired bootstrap. Reuses the teachers and final-arm runs already trained |
| E1/E2 teacher seed 52/62 | final arm x 3 seeds | is 0.9186 reachable with other teacher seeds? teacher variance |
| E3-E6 split variants c1p0..c4p0 | final arm x 3 seeds | which single validation image moved to train? (historical IDs are unknown) |
| Grad-CAM, external, summary | Reviewer 2 | heat maps, external dataset (needs your data), cross-experiment table vs the paper |

Run cells in order. Cells 3-5 are independent, long and can be stopped at any time; the summary cell reports whatever finished.

In [ ]:
# 1) Setup: Drive, code, packages, GPU
import os, sys, subprocess, zipfile, shutil
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DR_Research'
WORK = '/content/colab_retrain_comprehensive'
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
if os.path.isfile(DRIVE_ROOT + '/colab_retrain_comprehensive/dr_common.py'):
    shutil.copytree(DRIVE_ROOT + '/colab_retrain_comprehensive', WORK)
else:
    shutil.rmtree('/content/_unz', ignore_errors=True)
    with zipfile.ZipFile(DRIVE_ROOT + '/colab_retrain_comprehensive.zip') as z:
        print('ZIP entries:', z.namelist()[:5])
        z.extractall('/content/_unz')
    hits = [r for r, _, fs in os.walk('/content/_unz') if 'dr_common.py' in fs]
    assert hits, 'dr_common.py is not inside the ZIP'
    shutil.copytree(hits[0], WORK)

os.chdir(WORK); sys.path.insert(0, WORK)
assert os.path.isfile(WORK + '/reference/split_manifest_original.json'), 'reference/ folder missing (ZIP built with backslashes?)'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], capture_output=True, text=True, cwd='/content')
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-1500:])
import importlib; importlib.invalidate_caches()

os.makedirs('/content/data', exist_ok=True)
for f in ['train.csv', 'cache_224.npz']:
    if not os.path.exists('/content/data/' + f):
        shutil.copy(f'{DRIVE_ROOT}/data/{f}', '/content/data/' + f)
os.environ['DATA_DIR'] = '/content/data'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU runtime!')
assert torch.cuda.is_available(), 'No GPU'
du = shutil.disk_usage('/content/drive')
print('Drive free GB: %.1f / total %.1f' % (du.free / 1e9, du.total / 1e9), '| local free GB: %.1f' % (shutil.disk_usage('/content').free / 1e9))

In [ ]:
# 2) Helper: one resumable experiment = teachers -> students -> analysis (mirrored to Drive)
import json, glob, time, numpy as np, pandas as pd, dr_common as C
RES_LOCAL = '/content/results'
RES_DRIVE = DRIVE_ROOT + '/results_retrain_comprehensive'
OLD_RUN = DRIVE_ROOT + '/results_retrain/aptos_manuscript_final_split366_20261012_v1'   # earlier final-arm run (teachers + 3 students)
FINAL = 'teacher_equallogit__student_equal'
ALL_ARMS = [FINAL, 'teacher_equallogit__student_gated', 'teacher_equalprob__student_equal', 'teacher_equalprob__student_gated',
            'teacher_gate__student_equal', 'teacher_gate__student_gated', 'teacher_t1__student_equal', 'teacher_t1__student_gated',
            'teacher_t2__student_equal', 'teacher_t2__student_gated', 'teacher_gate__student_gated_aug', 'teacher_equallogit__student_equal_aug']

def _drop_resume_files(root):
    # finished runs keep best_model.pth + metrics; resume checkpoints only waste Drive quota
    for f in glob.glob(root + '/**/checkpoint_current.pth', recursive=True):
        d = os.path.dirname(f)
        if os.path.exists(d + '/best_model.pth') and os.path.exists(d + '/test_metrics.json'):
            os.remove(f)

def sh(cmd, env=None):
    # stream child output into the cell (plain subprocess.run output is not shown in Colab)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

def run_exp(name, split_move='c0p0', teacher_seed=42, arms=(FINAL,), seeds='42,52,62', analyze=True, drive_dir=None):
    local, drive = f'{RES_LOCAL}/{name}', drive_dir or f'{RES_DRIVE}/{name}'
    if os.path.isdir(drive):
        shutil.copytree(drive, local, dirs_exist_ok=True)   # restore after a runtime reset
    env = dict(os.environ, OUT_DIR=local, DRIVE_DIR=drive, SPLIT_MOVE=split_move, ARMS=','.join(arms), SEEDS=seeds, BATCH='16')
    env.pop('DR_SMOKE', None); env.pop('DR_MAX_EPOCHS', None)
    stages = [('teachers', ['--seed', str(teacher_seed)]), ('students', [])] + ([('analyze', [])] if analyze else [])
    t0 = time.time()
    for st, extra in stages:
        rc = sh([sys.executable, '-u', '-W', 'ignore', 'run_all.py', '--stage', st, *extra], env)
        assert rc == 0, f'{name}: stage {st} failed'
    _drop_resume_files(local); _drop_resume_files(drive)
    C.sync_dirs(local, drive)
    print(f'== {name} done in {(time.time()-t0)/60:.1f} min -> {drive}')

# sanity: split variants keep test IDs and move exactly one validation image
for mv in ['c0p0', 'c1p0', 'c2p0', 'c3p0', 'c4p0']:
    os.environ['SPLIT_MOVE'] = mv
    import importlib; importlib.reload(C)
    d = C.get_data('/content/data', batch_size=16)
    print(mv, {k: len(v) for k, v in d['ids'].items()}, 'class weights', {k: round(v, 3) for k, v in d['class_weights'].items()})
os.environ['SPLIT_MOVE'] = 'c0p0'; importlib.reload(C)

In [ ]:
# 3) E0_main - highest priority. Full 12-arm matrix on the manuscript-count split (existing teachers/final-arm runs are reused and skipped).
#    Covers: gate vs equal (+0.005 claim), T1/T2 roles, augmentation on the FINAL model, latency, calibration, bootstrap CIs.
run_exp('E0_main', 'c0p0', 42, ALL_ARMS, drive_dir=OLD_RUN)

In [ ]:
# 4) E1/E2 - teacher-seed robustness of the final model (can the paper's 0.9186 appear with another teacher seed?)
for ts in (52, 62):
    run_exp(f'E_teacherseed{ts}', 'c0p0', ts, (FINAL,))

In [ ]:
# 5) E3-E6 - which single validation image was moved? (historical manuscript split IDs are unknown) - final arm x 3 seeds each
for mv in ('c1p0', 'c2p0', 'c3p0', 'c4p0'):
    run_exp(f'E_split_{mv}', mv, 42, (FINAL,))

In [ ]:
# 6) Grad-CAM (Reviewer 2): final model and the gated comparison model, seed 42, 3 images per grade
from IPython.display import Image, display
for arm in (FINAL, 'teacher_gate__student_gated'):
    env = dict(os.environ, OUT_DIR=f'{RES_LOCAL}/E0_main', ARM=arm, SEED='42', N_PER_CLASS='3', SPLIT_MOVE='c0p0')
    rc = sh([sys.executable, '-u', '-W', 'ignore', 'gradcam.py'], env)
    p = f'{RES_LOCAL}/E0_main/gradcam/gradcam_{arm}_seed42.png'
    if rc == 0 and os.path.exists(p):
        display(Image(p))
    else:
        print('Grad-CAM failed/missing for', arm)
C.sync_dirs(f'{RES_LOCAL}/E0_main', OLD_RUN)

In [ ]:
# 7) OPTIONAL external dataset evaluation (evaluation only). Needs your own labelled data in Drive:
#    external/labels.csv (id_code,diagnosis 0-4) + external/images/ (files start with id_code). Good candidates: IDRiD, Messidor-2, EyePACS.
RUN_EXTERNAL = False
EXT = DRIVE_ROOT + '/external'
if RUN_EXTERNAL:
    for arm in (FINAL, 'teacher_gate__student_gated'):
        assert sh([sys.executable, '-u', 'external_eval.py', '--csv', EXT + '/labels.csv', '--images', EXT + '/images',
                   '--out', f'{RES_LOCAL}/E0_main/external_{arm}', '--results-root', f'{RES_LOCAL}/E0_main', '--arm', arm]) == 0
    C.sync_dirs(f'{RES_LOCAL}/E0_main', OLD_RUN)

In [ ]:
# 8) Cross-experiment summary vs the manuscript (QWK 0.9186+-0.0010, Acc 85.47, F1 0.703) + key E0 analysis blocks
for n in os.listdir(RES_DRIVE) if os.path.isdir(RES_DRIVE) else []:
    if not os.path.isdir(f'{RES_LOCAL}/{n}'):
        shutil.copytree(f'{RES_DRIVE}/{n}', f'{RES_LOCAL}/{n}')
assert sh([sys.executable, '-u', 'summarize_all.py', RES_LOCAL, RES_LOCAL + '/_summary']) == 0
rep_path = RES_LOCAL + '/E0_main/analysis/report.json'
if os.path.exists(rep_path):
    rep = json.load(open(rep_path))
    for k in ('inference_gate_vs_equal_swap', 'retrained_gate_vs_equal', 'augmentation_vs_no_augmentation', 'system_vs_student_only', 'latency_and_params'):
        print(k, json.dumps(rep.get(k), indent=1)[:1500])
os.makedirs(RES_DRIVE + '/_summary', exist_ok=True)
C.sync_dirs(RES_LOCAL + '/_summary', RES_DRIVE + '/_summary')
print('summary mirrored to', RES_DRIVE + '/_summary')